# Sales Performance Dashboard

## Objective
Analyze sales performance data to identify trends, patterns, and actionable insights for business decision-making.

## Analysis Includes:
1. Data Loading and Cleaning
2. Exploratory Data Analysis (EDA)
3. Sales Trends Analysis
4. Product Performance
5. Geographic Analysis
6. Visualizations and Insights

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Libraries imported successfully!")

## 1. Data Loading and Initial Exploration

In [ ]:
# Load datasets
sales_df = pd.read_csv('../data/sales_data.csv')
customer_df = pd.read_csv('../data/customer_data.csv')

# Convert date columns
sales_df['transaction_date'] = pd.to_datetime(sales_df['transaction_date'])
customer_df['registration_date'] = pd.to_datetime(customer_df['registration_date'])

print(f"Sales Data Shape: {sales_df.shape}")
print(f"Customer Data Shape: {customer_df.shape}")
print("\nData loaded successfully!")

In [ ]:
# Display first few rows
print("Sales Data Sample:")
display(sales_df.head())

print("\nCustomer Data Sample:")
display(customer_df.head())

In [ ]:
# Data info and summary statistics
print("=== Sales Data Info ===")
print(sales_df.info())
print("\n=== Sales Data Statistics ===")
print(sales_df.describe())

In [ ]:
# Check for missing values
print("Missing Values in Sales Data:")
print(sales_df.isnull().sum())
print("\nMissing Values in Customer Data:")
print(customer_df.isnull().sum())

## 2. Key Performance Indicators (KPIs)

In [ ]:
# Calculate key metrics
total_revenue = sales_df['revenue'].sum()
total_transactions = sales_df['transaction_id'].nunique()
unique_customers = sales_df['customer_id'].nunique()
avg_transaction_value = sales_df['revenue'].mean()
total_units_sold = sales_df['quantity'].sum()

# Display KPIs
print("="*50)
print("KEY PERFORMANCE INDICATORS")
print("="*50)
print(f"Total Revenue:              ${total_revenue:,.2f}")
print(f"Total Transactions:         {total_transactions:,}")
print(f"Unique Customers:           {unique_customers:,}")
print(f"Average Transaction Value:  ${avg_transaction_value:,.2f}")
print(f"Total Units Sold:           {total_units_sold:,}")
print(f"Avg Units per Transaction:  {total_units_sold/total_transactions:.2f}")
print("="*50)

## 3. Time Series Analysis

In [ ]:
# Daily sales trend
daily_sales = sales_df.groupby(sales_df['transaction_date'].dt.date).agg({
    'revenue': 'sum',
    'transaction_id': 'nunique',
    'customer_id': 'nunique'
}).reset_index()

daily_sales.columns = ['date', 'revenue', 'transactions', 'customers']
daily_sales['date'] = pd.to_datetime(daily_sales['date'])

# Create visualization
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Revenue trend
axes[0].plot(daily_sales['date'], daily_sales['revenue'], linewidth=2, color='#2E86AB')
axes[0].set_title('Daily Revenue Trend', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Revenue ($)')
axes[0].grid(True, alpha=0.3)

# Transaction count trend
axes[1].plot(daily_sales['date'], daily_sales['transactions'], linewidth=2, color='#A23B72')
axes[1].set_title('Daily Transaction Count', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Number of Transactions')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../visualizations/daily_trends.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Monthly sales trend
sales_df['year_month'] = sales_df['transaction_date'].dt.to_period('M')

monthly_sales = sales_df.groupby('year_month').agg({
    'revenue': 'sum',
    'transaction_id': 'nunique',
    'customer_id': 'nunique'
}).reset_index()

monthly_sales.columns = ['month', 'revenue', 'transactions', 'customers']
monthly_sales['month'] = monthly_sales['month'].astype(str)

# Calculate month-over-month growth
monthly_sales['revenue_growth'] = monthly_sales['revenue'].pct_change() * 100

print("Monthly Sales Performance:")
display(monthly_sales)

In [ ]:
# Visualize monthly trend with growth rate
fig, ax1 = plt.subplots(figsize=(14, 6))

# Bar chart for revenue
ax1.bar(monthly_sales['month'], monthly_sales['revenue'], color='#2E86AB', alpha=0.7, label='Revenue')
ax1.set_xlabel('Month', fontsize=12)
ax1.set_ylabel('Revenue ($)', fontsize=12)
ax1.tick_params(axis='y')
ax1.tick_params(axis='x', rotation=45)

# Line chart for growth rate
ax2 = ax1.twinx()
ax2.plot(monthly_sales['month'], monthly_sales['revenue_growth'], color='#F18F01', 
         marker='o', linewidth=2, markersize=8, label='MoM Growth %')
ax2.set_ylabel('Growth Rate (%)', fontsize=12)
ax2.axhline(y=0, color='red', linestyle='--', alpha=0.3)

plt.title('Monthly Revenue and Growth Rate', fontsize=14, fontweight='bold')
fig.legend(loc='upper left', bbox_to_anchor=(0.1, 0.9))
plt.tight_layout()
plt.savefig('../visualizations/monthly_revenue_growth.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Product Category Analysis

In [ ]:
# Product category performance
category_performance = sales_df.groupby('product_category').agg({
    'revenue': 'sum',
    'transaction_id': 'nunique',
    'quantity': 'sum',
    'customer_id': 'nunique'
}).reset_index()

category_performance.columns = ['category', 'revenue', 'transactions', 'units_sold', 'customers']
category_performance['avg_transaction_value'] = category_performance['revenue'] / category_performance['transactions']
category_performance['revenue_share'] = (category_performance['revenue'] / category_performance['revenue'].sum() * 100)

category_performance = category_performance.sort_values('revenue', ascending=False)

print("Product Category Performance:")
display(category_performance)

In [ ]:
# Visualize category performance
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Revenue by category (bar chart)
axes[0].barh(category_performance['category'], category_performance['revenue'], color='#2E86AB')
axes[0].set_xlabel('Revenue ($)', fontsize=12)
axes[0].set_title('Revenue by Product Category', fontsize=14, fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)

# Market share (pie chart)
axes[1].pie(category_performance['revenue'], labels=category_performance['category'], 
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 10})
axes[1].set_title('Revenue Share by Category', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../visualizations/category_performance.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Top Products Analysis

In [ ]:
# Top 10 products by revenue
top_products = sales_df.groupby(['product_name', 'product_category']).agg({
    'revenue': 'sum',
    'quantity': 'sum',
    'transaction_id': 'nunique'
}).reset_index()

top_products.columns = ['product', 'category', 'revenue', 'units_sold', 'transactions']
top_products = top_products.sort_values('revenue', ascending=False).head(10)

print("Top 10 Products by Revenue:")
display(top_products)

# Visualize
plt.figure(figsize=(12, 6))
bars = plt.barh(top_products['product'], top_products['revenue'])

# Color by category
colors = plt.cm.Set3(range(len(top_products)))
for bar, color in zip(bars, colors):
    bar.set_color(color)

plt.xlabel('Revenue ($)', fontsize=12)
plt.title('Top 10 Products by Revenue', fontsize=14, fontweight='bold')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../visualizations/top_products.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Store Location Performance

In [ ]:
# Store performance analysis
store_performance = sales_df.groupby('store_location').agg({
    'revenue': 'sum',
    'transaction_id': 'nunique',
    'customer_id': 'nunique'
}).reset_index()

store_performance.columns = ['store', 'revenue', 'transactions', 'customers']
store_performance['avg_transaction_value'] = store_performance['revenue'] / store_performance['transactions']
store_performance = store_performance.sort_values('revenue', ascending=False)

print("Store Location Performance:")
display(store_performance)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Revenue by store
axes[0].bar(store_performance['store'], store_performance['revenue'], color='#2E86AB')
axes[0].set_ylabel('Revenue ($)', fontsize=12)
axes[0].set_title('Revenue by Store Location', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)

# Average transaction value by store
axes[1].bar(store_performance['store'], store_performance['avg_transaction_value'], color='#F18F01')
axes[1].set_ylabel('Average Transaction Value ($)', fontsize=12)
axes[1].set_title('Avg Transaction Value by Store', fontsize=14, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../visualizations/store_performance.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Customer Segment Analysis

In [ ]:
# Merge sales with customer data
sales_customer = sales_df.merge(customer_df, on='customer_id', how='left')

# Segment performance
segment_performance = sales_customer.groupby('customer_segment').agg({
    'revenue': 'sum',
    'transaction_id': 'nunique',
    'customer_id': 'nunique'
}).reset_index()

segment_performance.columns = ['segment', 'revenue', 'transactions', 'customers']
segment_performance['avg_revenue_per_customer'] = segment_performance['revenue'] / segment_performance['customers']
segment_performance['avg_transaction_value'] = segment_performance['revenue'] / segment_performance['transactions']

segment_performance = segment_performance.sort_values('revenue', ascending=False)

print("Customer Segment Performance:")
display(segment_performance)

In [ ]:
# Visualize segment performance
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Revenue by segment
axes[0, 0].bar(segment_performance['segment'], segment_performance['revenue'], color='#2E86AB')
axes[0, 0].set_ylabel('Revenue ($)')
axes[0, 0].set_title('Revenue by Customer Segment', fontweight='bold')
axes[0, 0].tick_params(axis='x', rotation=45)
axes[0, 0].grid(axis='y', alpha=0.3)

# Customer count by segment
axes[0, 1].bar(segment_performance['segment'], segment_performance['customers'], color='#F18F01')
axes[0, 1].set_ylabel('Number of Customers')
axes[0, 1].set_title('Customer Count by Segment', fontweight='bold')
axes[0, 1].tick_params(axis='x', rotation=45)
axes[0, 1].grid(axis='y', alpha=0.3)

# Avg revenue per customer
axes[1, 0].bar(segment_performance['segment'], segment_performance['avg_revenue_per_customer'], color='#A23B72')
axes[1, 0].set_ylabel('Avg Revenue per Customer ($)')
axes[1, 0].set_title('Avg Revenue per Customer by Segment', fontweight='bold')
axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].grid(axis='y', alpha=0.3)

# Avg transaction value
axes[1, 1].bar(segment_performance['segment'], segment_performance['avg_transaction_value'], color='#18A558')
axes[1, 1].set_ylabel('Avg Transaction Value ($)')
axes[1, 1].set_title('Avg Transaction Value by Segment', fontweight='bold')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../visualizations/segment_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Day of Week & Hourly Analysis

In [ ]:
# Day of week analysis
sales_df['day_of_week'] = sales_df['transaction_date'].dt.day_name()
sales_df['day_num'] = sales_df['transaction_date'].dt.dayofweek

dow_analysis = sales_df.groupby(['day_num', 'day_of_week']).agg({
    'revenue': 'sum',
    'transaction_id': 'nunique'
}).reset_index().sort_values('day_num')

dow_analysis.columns = ['day_num', 'day', 'revenue', 'transactions']

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Revenue by day of week
axes[0].bar(dow_analysis['day'], dow_analysis['revenue'], color='#2E86AB')
axes[0].set_ylabel('Revenue ($)', fontsize=12)
axes[0].set_title('Revenue by Day of Week', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)

# Transactions by day of week
axes[1].bar(dow_analysis['day'], dow_analysis['transactions'], color='#F18F01')
axes[1].set_ylabel('Number of Transactions', fontsize=12)
axes[1].set_title('Transactions by Day of Week', fontsize=14, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../visualizations/day_of_week_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Day of Week Performance:")
display(dow_analysis[['day', 'revenue', 'transactions']])

## 9. Key Insights and Recommendations

In [ ]:
# Generate insights summary
print("="*70)
print("KEY INSIGHTS AND RECOMMENDATIONS")
print("="*70)

print("\n1. REVENUE INSIGHTS:")
print(f"   - Total Revenue: ${total_revenue:,.2f}")
print(f"   - Best performing category: {category_performance.iloc[0]['category']}")
print(f"     (${category_performance.iloc[0]['revenue']:,.2f}, {category_performance.iloc[0]['revenue_share']:.1f}% of total)")

print("\n2. STORE PERFORMANCE:")
best_store = store_performance.iloc[0]
print(f"   - Top performing store: {best_store['store']}")
print(f"     (${best_store['revenue']:,.2f} revenue, {best_store['transactions']:,} transactions)")

print("\n3. CUSTOMER SEGMENTS:")
best_segment = segment_performance.iloc[0]
print(f"   - Highest revenue segment: {best_segment['segment']}")
print(f"     (${best_segment['avg_revenue_per_customer']:,.2f} per customer)")

print("\n4. PRODUCT INSIGHTS:")
print(f"   - Top selling product: {top_products.iloc[0]['product']}")
print(f"     (${top_products.iloc[0]['revenue']:,.2f} total revenue)")

print("\n5. RECOMMENDATIONS:")
print("   - Focus marketing efforts on Premium and Regular customer segments")
print("   - Optimize inventory for top-performing products")
print("   - Replicate successful strategies from top-performing stores")
print("   - Consider promotional campaigns during slower days/periods")
print("   - Implement cross-selling strategies for high-value categories")
print("="*70)

## Conclusion

This analysis provides comprehensive insights into sales performance across multiple dimensions:
- Temporal trends (daily, monthly, day of week)
- Product and category performance
- Geographic distribution (store locations)
- Customer segmentation

The findings can be used to:
1. Optimize inventory management
2. Target marketing campaigns
3. Improve store operations
4. Enhance customer retention strategies
5. Drive revenue growth

**Next Steps:**
- Implement predictive models for sales forecasting
- Conduct deeper customer behavior analysis
- Perform cohort analysis for customer retention
- Create interactive dashboards for real-time monitoring